# Day 12 - Raw to Cleaned Mini Challenge

**Topic:** Raw to Cleaned Mini Challenge  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** รวม skill จาก Day 01-11 เพื่อแปลง messy raw source data ให้เป็น cleaned dataset พร้อม DQ flags, deduplication และ Lakehouse table output

วันนี้เป็น mini challenge แรกของ PySpark Lakehouse Challenge เราจะเอาสิ่งที่เรียนมาก่อนหน้า เช่น schema, column operations, null handling, date parsing, string cleansing, nested JSON, window deduplication และ built-in functions มาประกอบเป็น raw-to-cleaned transformation flow แบบเล็ก ๆ ที่ใกล้เคียงงาน Data Engineering จริง

## Goal of the day

1. รวม pattern ที่เรียนจาก Day 01-11 มาใช้กับ messy raw dataset ชุดเดียว
2. อธิบาย flow จาก raw data ไปเป็น cleaned data ได้แบบเป็นขั้นตอน
3. ใช้ PySpark เพื่อ standardize column names, parse types, clean strings และ flatten nested JSON ได้
4. สร้าง DQ flag columns เพื่อแยก clean records กับ rejected records ได้
5. ใช้ window function เพื่อ keep latest record ต่อ business key และจัดการ duplicate updates ได้
6. เขียน output เป็น Lakehouse tables สำหรับ cleaned และ rejected records ได้เมื่อ attach Lakehouse เรียบร้อย

## Why it matters in production

ใน production pipeline เราแทบไม่เคยได้ข้อมูลที่ clean ตั้งแต่ต้นทาง ข้อมูล raw มักมีปัญหาเช่น:

- column names ไม่สม่ำเสมอ เช่น มี space, uppercase/lowercase ปนกัน
- field สำคัญเป็น `null` หรือ blank string
- date / timestamp มาได้หลาย format
- amount เป็น string ที่มี comma, currency symbol หรือค่าที่ cast ไม่ได้
- status มี case หรือ spelling ไม่สม่ำเสมอ
- nested JSON บาง record parse ได้ บาง record parse ไม่ได้
- source ส่ง record ซ้ำหรือ late update เข้ามา

ถ้า pipeline clean ข้อมูลโดยไม่มี DQ flag หรือ reject output เราจะ debug ยากมากว่าข้อมูลหายไปตรงไหน และ record ไหนถูกตัดออกเพราะอะไร

Production takeaway:

> Raw-to-cleaned ไม่ใช่แค่ `select` แล้ว `cast`; ต้องมองเห็น data issue, duplicate logic และ output ที่ตรวจสอบย้อนหลังได้

## Key concepts

| Concept | Meaning |
|---|---|
| Raw data | ข้อมูลต้นทางที่ยังไม่ standardize และอาจมี dirty values |
| Cleaned data | ข้อมูลที่ผ่านการ normalize, cast, parse, deduplicate และ basic DQ แล้ว |
| Business key | key ที่ใช้ระบุ record ตาม business logic เช่น `transaction_id` |
| Type casting | การเปลี่ยน data type เช่น string amount เป็น DoubleType, string date เป็น DateType |
| Data standardization | การทำ format ให้ consistent เช่น trim, lower, upper, remove special characters |
| Nested JSON parsing | การใช้ `from_json` และ explicit schema เพื่อแปลง JSON string เป็น struct |
| Corrupt JSON handling | การใช้ `PERMISSIVE` mode ร่วมกับ `_corrupt_record` ใน `from_json` เพื่อเก็บ malformed JSON payload แทนการปล่อยให้กลายเป็น null fields เงียบ ๆ |
| DQ flag | column ที่บอกว่า record มี issue อะไรบ้าง |
| Reject / quarantine mindset | ไม่ drop bad records เงียบ ๆ แต่แยกออกมาให้ตรวจสอบได้ |
| Latest-record deduplication | การใช้ window function เพื่อเลือก record ล่าสุดในแต่ละ key |


## Concept explanation

### Raw-to-cleaned flow คืออะไร?

Raw-to-cleaned flow คือขั้นตอนที่เปลี่ยนข้อมูลจาก source format ให้กลายเป็น dataset ที่พร้อมใช้งานต่อใน downstream transformation หรือ reporting มากขึ้น

ใน notebook นี้เราจะทำ flow แบบง่าย ๆ:

1. รับ raw DataFrame ที่มีปัญหาหลายแบบ
2. ทำ column name normalization
3. standardize string fields
4. parse amount, date และ timestamp
5. parse nested JSON payload
6. สร้าง DQ issues ต่อ record
7. deduplicate ด้วย `transaction_id` และ `updated_at`
8. แยก output เป็น cleaned records และ rejected records
9. write output เป็น Lakehouse tables

### Cleaned ไม่ได้แปลว่า perfect

คำว่า cleaned ใน data pipeline ไม่ได้แปลว่าข้อมูลสมบูรณ์ 100% แต่แปลว่า:

- schema คาดเดาได้มากขึ้น
- data type เหมาะกับการใช้งานมากขึ้น
- string/date/amount ถูก standardize เท่าที่จำเป็น
- invalid records ถูก identify ได้
- duplicate logic ถูกใช้ชัดเจน
- output สามารถตรวจสอบ row count และ issue summary ได้

> Cleaned dataset ที่ดีควร explain ได้ว่า record ไหนผ่าน record ไหนไม่ผ่าน และเพราะอะไร

### ทำไมต้องแยก rejected records?

ถ้าเราเขียน code แบบนี้:

```python
df_clean = df.filter("amount > 0")
```

ข้อมูลที่ amount ไม่ถูกต้องจะหายไปทันทีโดยไม่มี evidence ว่าหายเพราะอะไร ในงานจริงควรมี reject / quarantine output อย่างน้อยในรูปแบบง่าย ๆ เช่น:

- original key
- source system
- raw values ที่เกี่ยวข้อง
- DQ issue list
- processing timestamp

Day 12 ยังไม่ทำ quarantine pattern แบบเต็ม เพราะจะมี Day 22 แยกต่างหาก แต่วันนี้จะเริ่มฝึก mindset ว่า bad records ต้องมองเห็นได้


## Mermaid diagram: Raw to Cleaned Flow

```mermaid
flowchart LR
    A[Messy Raw Source] --> B[Normalize Column Names]
    B --> C[Standardize Strings and Parse Types]
    C --> D[Parse Nested JSON]
    D --> E[Create DQ Issue Flags]
    E --> F[Deduplicate by Business Key]
    F --> G{Clean?}
    G -- Yes --> H[Cleaned Transactions Table]
    G -- No --> I[Rejected Records Table]
    H --> J[Daily Summary / Downstream Use]
```

Key idea:

> Pipeline ที่ดีไม่ควรมีแค่ clean output แต่ควรมี rejected output เพื่อให้เห็นว่า data issue เกิดจากอะไร

## Setup / imports

In [1]:
import re  # Regular expression utilities

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 3, Finished, Available, Finished, False)

## Create mock data

Dataset นี้ตั้งใจทำให้ messy เพื่อฝึก raw-to-cleaned transformation จริง ๆ เช่น:

- column names ใน raw schema มี space
- `amount` เป็น string หลาย format
- `transaction_date` มีหลาย format และ invalid date
- `status` มี uppercase/lowercase/blank space ปนกัน
- `source_payload_json` มีทั้ง valid JSON, missing payload และ malformed JSON
- มี duplicate transaction ที่ต้องเลือก latest record ด้วย `updated_at`

In [2]:
raw_transactions_data = [
    (
        "T001", "001", "Alice Wong", " Alice@example.com ", "+66 81-111-1111",
        "2026-05-01", "1,200.50", " success ", "Credit Card",
        '{"device":{"platform":"ios","app_version":"1.2.0"},"location":{"city":"Bangkok","country":"TH"},"promo":{"code":"WELCOME10"}}',
        "2026-05-01 10:00:00", "mobile_app", "BATCH_001"
    ),
    (
        "T001", "001", "Alice Wong", "alice@example.com", "+66 81 111 1111",
        "2026-05-01", "1,150.00", "SUCCESS", "credit_card",
        '{"device":{"platform":"ios","app_version":"1.1.0"},"location":{"city":"Bangkok","country":"TH"},"promo":{"code":"WELCOME10"}}',
        "2026-05-01 09:00:00", "mobile_app", "BATCH_001"
    ),
    (
        "T002", "002", "Bob Lee", "bob@example.com", "0812222222",
        "01/05/2026", "0", "success", "PromptPay",
        '{"device":{"platform":"android","app_version":"2.0.0"},"location":{"city":"Chiang Mai","country":"TH"},"promo":{"code":null}}',
        "2026-05-01 11:00:00", "mobile_app", "BATCH_001"
    ),
    (
        "T003", "003", "Charlie Kim", "charlie@example.com", "0813333333",
        "invalid_date", "450.00", "SUCCESS", "bank transfer",
        '{"device":{"platform":"web","app_version":"3.1.0"},"location":{"city":"Rayong","country":"TH"},"promo":{"code":"RAYONG"}}',
        "2026-05-01 12:00:00", "web_portal", "BATCH_001"
    ),
    (
        "T004", None, "Dana Lim", "dana@example.com", "0814444444",
        "2026-05-02", "-25.00", "refunded", "credit_card",
        '{"device":{"platform":"ios","app_version":"1.2.0"},"location":{"city":"Phuket","country":"TH"},"promo":{"code":null}}',
        "2026-05-02T08:30:00", "mobile_app", "BATCH_002"
    ),
    (
        "", "005", "Eve Tan", "eve@example.com", "0815555555",
        "2026-05-02", "890.00", "pending", "promptpay",
        '{"device":{"platform":"android","app_version":"2.0.1"},"location":{"city":"Bangkok","country":"TH"},"promo":{"code":"MAY"}}',
        "2026-05-02 09:15:00", "mobile_app", "BATCH_002"
    ),
    (
        "T006", "006", "Frank Ho", "frank@example.com", "0816666666",
        "2026/05/03", "780.25", "unknown", "cash",
        '{"device":{"platform":"pos","app_version":"1.0.0"},"location":{"city":"Khon Kaen","country":"TH"},"promo":{"code":null}}',
        "2026-05-03 10:00:00", "pos", "BATCH_002"
    ),
    (
        "T007", "007", "Grace Park", "grace@example.com", "0817777777",
        "2026-05-03", "3,200.00", "success", "credit card",
        "not_json",
        "2026-05-03 10:30:00", "web_portal", "BATCH_002"
    ),
    (
        "T008", "008", "Henry Yu", "henry@example.com", "0818888888",
        "2026-05-04", "THB 2,500.00", "success", "bank_transfer",
        '{"device":{"platform":"web","app_version":"3.2.0"},"location":{"city":"Bangkok","country":"TH"},"promo":{"code":"VIP"}}',
        "2026-05-04 08:00:00", "web_portal", "BATCH_003"
    ),
    (
        "T009", "009", "Ivy Chen", "ivy@example.com", "0819999999",
        "2026-05-04", "not_available", "failed", "credit_card",
        '{"device":{"platform":"ios","app_version":"1.2.1"},"location":{"city":"Bangkok","country":"TH"},"promo":{"code":null}}',
        "2026-05-04 08:30:00", "mobile_app", "BATCH_003"
    ),
    (
        "T010", "010", "Jack Ma", "jack@example.com", "0820000000",
        "2026-05-05", "1000", " Success ", "credit card",
        None,
        "2026-05-05 09:00:00", "call_center", "BATCH_003"
    ),
    (
        "t011", "C-011", "Kate Lin", " KATE@example.com ", "0821111111",
        "2026-05-05", "500", "success", "PromptPay",
        '{"device":{"platform":"android","app_version":"2.1.0"},"location":{"city":"Chiang Mai","country":"TH"},"promo":{"code":"CM"}}',
        "2026-05-05 09:30:00", "mobile_app", "BATCH_003"
    ),
]

raw_schema = T.StructType([
    T.StructField("Transaction ID", T.StringType(), True),
    T.StructField("Customer ID", T.StringType(), True),
    T.StructField("Customer Name", T.StringType(), True),
    T.StructField("Email", T.StringType(), True),
    T.StructField("Phone", T.StringType(), True),
    T.StructField("Transaction Date", T.StringType(), True),
    T.StructField("Amount", T.StringType(), True),
    T.StructField("Status", T.StringType(), True),
    T.StructField("Payment Method", T.StringType(), True),
    T.StructField("Source Payload JSON", T.StringType(), True),
    T.StructField("Updated At", T.StringType(), True),
    T.StructField("Source System", T.StringType(), True),
    T.StructField("Batch ID", T.StringType(), True),
])

df_raw = spark.createDataFrame(raw_transactions_data, raw_schema)

df_raw.show(truncate=False)
df_raw.printSchema()
print("Raw row count:", df_raw.count())

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 4, Finished, Available, Finished, False)

+--------------+-----------+-------------+-------------------+---------------+----------------+-------------+---------+--------------+-----------------------------------------------------------------------------------------------------------------------------+-------------------+-------------+---------+
|Transaction ID|Customer ID|Customer Name|Email              |Phone          |Transaction Date|Amount       |Status   |Payment Method|Source Payload JSON                                                                                                          |Updated At         |Source System|Batch ID |
+--------------+-----------+-------------+-------------------+---------------+----------------+-------------+---------+--------------+-----------------------------------------------------------------------------------------------------------------------------+-------------------+-------------+---------+
|T001          |001        |Alice Wong   | Alice@example.com |+66 81-111-1111|2026-05

## PySpark code examples

ใน section นี้จะทำ raw-to-cleaned flow แบบ end-to-end โดยตั้งใจให้ code แต่ละ step อ่านง่ายและตรวจ output ได้ก่อนขยับไป step ถัดไป

### Example 1: Normalize column names

Raw source หลายระบบชอบส่ง column names มาไม่เหมือนกัน เช่นมี space, uppercase/lowercase ปนกัน หรือมี special characters

ขั้นแรกเราจะเปลี่ยน column names ให้เป็น `snake_case` เพื่อให้เขียน PySpark expression ง่ายขึ้นและลดปัญหาเวลาทำ downstream transformation

In [3]:
def normalize_column_name(column_name: str) -> str:
    normalized = column_name.strip().lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized)  # Replace non-alphanumeric characters with underscore
    normalized = re.sub(r"_+", "_", normalized).strip("_")  # Collapse repeated underscores and trim edge underscores
    return normalized

cleaned_column_names = [normalize_column_name(column_name) for column_name in df_raw.columns]

# toDF() renames all columns using the specified column names
# * unpacks list into multiple column-name arguments
df_raw_named = df_raw.toDF(*cleaned_column_names)

df_raw_named.show(truncate=False)
print(df_raw_named.columns)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 5, Finished, Available, Finished, False)

+--------------+-----------+-------------+-------------------+---------------+----------------+-------------+---------+--------------+-----------------------------------------------------------------------------------------------------------------------------+-------------------+-------------+---------+
|transaction_id|customer_id|customer_name|email              |phone          |transaction_date|amount       |status   |payment_method|source_payload_json                                                                                                          |updated_at         |source_system|batch_id |
+--------------+-----------+-------------+-------------------+---------------+----------------+-------------+---------+--------------+-----------------------------------------------------------------------------------------------------------------------------+-------------------+-------------+---------+
|T001          |001        |Alice Wong   | Alice@example.com |+66 81-111-1111|2026-05

### Example 2: Standardize strings and parse basic data types

ก่อน cast data type ควร clean raw string ให้พอเหมาะก่อน เช่น:

- trim ช่องว่างหัวท้าย
- เปลี่ยน key/status ให้เป็น format เดียวกัน
- remove comma/currency symbol จาก amount
- parse date หลาย format ด้วย `coalesce`
- parse timestamp หลาย format ด้วย `coalesce`

จุดสำคัญคืออย่า cast แบบเร็วเกินไปถ้า raw value ยังมี format ปนกัน

In [4]:
def null_if_blank(column_name: str):
    return (
        F.when(F.length(F.trim(F.col(column_name))) == 0, F.lit(None))
         .otherwise(F.trim(F.col(column_name)))
    )

allowed_status = ["success", "failed", "pending", "refunded"]

amount_numeric_string = F.regexp_replace(null_if_blank("amount"), r"[^0-9.-]", "")  # Remove non-numeric characters except dot and minus sign

df_standardized = (
    df_raw_named
    .withColumn("transaction_id", F.upper(null_if_blank("transaction_id")))
    .withColumn("customer_id", F.regexp_replace(null_if_blank("customer_id"), r"[^0-9]", ""))
    .withColumn("customer_name", null_if_blank("customer_name"))
    .withColumn("email", F.lower(null_if_blank("email")))
    .withColumn("phone", F.regexp_replace(null_if_blank("phone"), r"[^0-9+]", ""))  # Remove non-numeric characters except plus sign
    .withColumn("status", F.lower(null_if_blank("status")))
    .withColumn("payment_method", F.lower(F.regexp_replace(null_if_blank("payment_method"), r"[\s-]+", "_")))  # Replace spaces and hyphens with underscore
    .withColumn("amount", amount_numeric_string.cast("double"))
    .withColumn(
        "transaction_date",
        F.coalesce(
            F.to_date("transaction_date", "yyyy-MM-dd"),
            F.to_date("transaction_date", "dd/MM/yyyy"),
            F.to_date("transaction_date", "yyyy/MM/dd"),
        )
    )
    .withColumn(
        "updated_at",
        F.coalesce(
            F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("updated_at", "yyyy-MM-dd'T'HH:mm:ss"),
        )
    )
)

df_standardized.select(
    "transaction_id",
    "customer_id",
    "email",
    "phone",
    "transaction_date",
    "amount",
    "status",
    "payment_method",
    "updated_at",
).show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 6, Finished, Available, Finished, False)

+--------------+-----------+-------------------+------------+----------------+------+--------+--------------+-------------------+
|transaction_id|customer_id|email              |phone       |transaction_date|amount|status  |payment_method|updated_at         |
+--------------+-----------+-------------------+------------+----------------+------+--------+--------------+-------------------+
|T001          |001        |alice@example.com  |+66811111111|2026-05-01      |1200.5|success |credit_card   |2026-05-01 10:00:00|
|T001          |001        |alice@example.com  |+66811111111|2026-05-01      |1150.0|success |credit_card   |2026-05-01 09:00:00|
|T002          |002        |bob@example.com    |0812222222  |2026-05-01      |0.0   |success |promptpay     |2026-05-01 11:00:00|
|T003          |003        |charlie@example.com|0813333333  |NULL            |450.0 |success |bank_transfer |2026-05-01 12:00:00|
|T004          |NULL       |dana@example.com   |0814444444  |2026-05-02      |-25.0 |refun

### Example 3: Parse nested JSON payload

ใน source หลายแบบ เช่น API events หรือ application logs ข้อมูลบางส่วนอาจถูกส่งมาเป็น JSON string ภายใน column เดียว

เราจะใช้ `from_json` พร้อม explicit schema เพื่อแปลง JSON string เป็น struct แล้ว flatten fields ที่จำเป็นออกมาเป็น columns

In [5]:
payload_schema = T.StructType([
    T.StructField(
        "device",
        T.StructType([
            T.StructField("platform", T.StringType(), True),
            T.StructField("app_version", T.StringType(), True),
        ]),
        True,
    ),
    T.StructField(
        "location",
        T.StructType([
            T.StructField("city", T.StringType(), True),
            T.StructField("country", T.StringType(), True),
        ]),
        True,
    ),
    T.StructField(
        "promo",
        T.StructType([
            T.StructField("code", T.StringType(), True),
        ]),
        True,
    ),
    T.StructField("_corrupt_record", T.StringType(), True),  # Store malformed JSON payload
])

# Capture malformed JSON explicitly because default parsing may turn invalid payloads into silent null fields.
json_parse_options = {
    "mode": "PERMISSIVE",  # Capture malformed JSON payloads without failing the job
    "columnNameOfCorruptRecord": "_corrupt_record",  # Store malformed JSON payload in this field
}

df_with_payload = (
    df_standardized
    .withColumn("payload", F.from_json(F.col("source_payload_json"), payload_schema, json_parse_options))
    .withColumn(
        "payload_parse_status",
        F.when(F.col("source_payload_json").isNull(), F.lit("missing_payload"))
         .when(F.col("payload._corrupt_record").isNotNull(), F.lit("invalid_json"))
         .otherwise(F.lit("parsed"))
    )
    .withColumn("device_platform", F.col("payload.device.platform"))
    .withColumn("device_app_version", F.col("payload.device.app_version"))
    .withColumn("city", F.col("payload.location.city"))
    .withColumn("country", F.col("payload.location.country"))
    .withColumn("promo_code", F.col("payload.promo.code"))
    .withColumn("payload_corrupt_record", F.col("payload._corrupt_record"))
)

df_with_payload.select(
    "transaction_id",
    "source_payload_json",
    "payload_parse_status",
    "payload_corrupt_record",
    "device_platform",
    "city",
    "country",
    "promo_code",
).show(truncate=False)

df_with_payload.printSchema()

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 7, Finished, Available, Finished, False)

+--------------+-----------------------------------------------------------------------------------------------------------------------------+--------------------+----------------------+---------------+----------+-------+----------+
|transaction_id|source_payload_json                                                                                                          |payload_parse_status|payload_corrupt_record|device_platform|city      |country|promo_code|
+--------------+-----------------------------------------------------------------------------------------------------------------------------+--------------------+----------------------+---------------+----------+-------+----------+
|T001          |{"device":{"platform":"ios","app_version":"1.2.0"},"location":{"city":"Bangkok","country":"TH"},"promo":{"code":"WELCOME10"}}|parsed              |NULL                  |ios            |Bangkok   |TH     |WELCOME10 |
|T001          |{"device":{"platform":"ios","app_version":"1.1.0"},"

### Example 4: Create DQ issue flags

แทนที่จะ filter records ทิ้งทันที เราจะสร้าง list ของ issue ต่อ record ก่อน เพื่อให้ตรวจสอบได้ว่า record ไม่ผ่านเพราะอะไร

ในตัวอย่างนี้เราจะเช็กแบบง่าย ๆ:

- missing `transaction_id`
- missing `customer_id`
- invalid `amount`
- invalid `transaction_date`
- invalid `updated_at`
- invalid `status`
- invalid JSON payload

In [ ]:
df_with_dq_columns = (
    df_with_payload
    .withColumn(
        "dq_missing_transaction_id",
        F.when(F.col("transaction_id").isNull(), F.lit("missing_transaction_id"))
    )
    .withColumn(
        "dq_missing_customer_id",
        F.when((F.col("customer_id").isNull()) | (F.length(F.col("customer_id")) == 0), F.lit("missing_customer_id"))
    )
    .withColumn(
        "dq_invalid_amount",
        F.when((F.col("amount").isNull()) | (F.col("amount") <= 0), F.lit("invalid_amount"))
    )
    .withColumn(
        "dq_invalid_transaction_date",
        F.when(F.col("transaction_date").isNull(), F.lit("invalid_transaction_date"))
    )
    .withColumn(
        "dq_invalid_updated_at",
        F.when(F.col("updated_at").isNull(), F.lit("invalid_updated_at"))
    )
    .withColumn(
        "dq_invalid_status",
        F.when((F.col("status").isNull()) | (~F.col("status").isin(allowed_status)), F.lit("invalid_status"))
    )
    .withColumn(
        "dq_invalid_payload",
        F.when(F.col("payload_parse_status") == "invalid_json", F.lit("invalid_payload_json"))
    )
)

df_with_dq = (
    df_with_dq_columns
    .withColumn(
        "dq_issues",
        F.expr("""
            filter(
                array(
                    dq_missing_transaction_id,
                    dq_missing_customer_id,
                    dq_invalid_amount,
                    dq_invalid_transaction_date,
                    dq_invalid_updated_at,
                    dq_invalid_status,
                    dq_invalid_payload
                ),
                x -> x is not null
            )
        """)
    )
    .withColumn(
        "dq_status",
        F.when(F.size(F.col("dq_issues")) == 0, F.lit("passed")).otherwise(F.lit("failed"))
    )
)

# Tips:
# - expr("SQL expression") lets you use Spark SQL functions inside PySpark DataFrame code.
# - array(value1, value2, ...) combines multiple values into one array.
# - Higher-order function: a function that applies logic to elements inside an array.
#   Example: filter(array_column, element -> condition) keeps only array elements that match the condition.
# - Lambda expression / anonymous function: inline logic defined without a separate function name.
#   Example: element -> condition defines the filtering logic for each array element.

df_with_dq.select(
    "transaction_id",
    "customer_id",
    "amount",
    "transaction_date",
    "status",
    "payload_parse_status",
    "dq_status",
    "dq_issues",
).show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 8, Finished, Available, Finished, False)

+--------------+-----------+------+----------------+--------+--------------------+---------+-------------------------------------+
|transaction_id|customer_id|amount|transaction_date|status  |payload_parse_status|dq_status|dq_issues                            |
+--------------+-----------+------+----------------+--------+--------------------+---------+-------------------------------------+
|T001          |001        |1200.5|2026-05-01      |success |parsed              |passed   |[]                                   |
|T001          |001        |1150.0|2026-05-01      |success |parsed              |passed   |[]                                   |
|T002          |002        |0.0   |2026-05-01      |success |parsed              |failed   |[invalid_amount]                     |
|T003          |003        |450.0 |NULL            |success |parsed              |failed   |[invalid_transaction_date]           |
|T004          |NULL       |-25.0 |2026-05-02      |refunded|parsed              |f

### Example 5: Deduplicate by latest update

สำหรับ record ที่มี business key เช่น `transaction_id` เราจะใช้ window function เพื่อเลือก latest record ตาม `updated_at`

ในตัวอย่างนี้:

- `dedup_rank = 1` คือ record ล่าสุดต่อ `transaction_id`
- `dedup_rank > 1` คือ older duplicate version
- record ที่ไม่มี `transaction_id` จะไม่ถูก deduplicate ด้วย key และจะถูก reject จาก DQ issue อยู่แล้ว

In [7]:
dedup_window = Window.partitionBy("transaction_id").orderBy(
    F.col("updated_at").desc_nulls_last(),
    F.col("batch_id").desc_nulls_last(),
)

df_with_key = df_with_dq.filter(F.col("transaction_id").isNotNull())
df_without_key = (
    df_with_dq
    .filter(F.col("transaction_id").isNull())
    .withColumn("dedup_rank", F.lit(None).cast("int"))  # Optional: cast("int") creates a null integer placeholder for clearer schema before unionByName()
)

df_ranked_with_key = df_with_key.withColumn("dedup_rank", F.row_number().over(dedup_window))

df_ranked = df_ranked_with_key.unionByName(df_without_key)

# Tips:
# - desc_nulls_last(): Sort newest values first and move null values to the end
# - unionByName(): Combine DataFrames by matching column names

df_ranked.select(
    "transaction_id",
    "amount",
    "updated_at",
    "batch_id",
    "dq_status",
    "dq_issues",
    "dedup_rank",
).orderBy("transaction_id", "dedup_rank").show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 9, Finished, Available, Finished, False)

+--------------+------+-------------------+---------+---------+-------------------------------------+----------+
|transaction_id|amount|updated_at         |batch_id |dq_status|dq_issues                            |dedup_rank|
+--------------+------+-------------------+---------+---------+-------------------------------------+----------+
|NULL          |890.0 |2026-05-02 09:15:00|BATCH_002|failed   |[missing_transaction_id]             |NULL      |
|T001          |1200.5|2026-05-01 10:00:00|BATCH_001|passed   |[]                                   |1         |
|T001          |1150.0|2026-05-01 09:00:00|BATCH_001|passed   |[]                                   |2         |
|T002          |0.0   |2026-05-01 11:00:00|BATCH_001|failed   |[invalid_amount]                     |1         |
|T003          |450.0 |2026-05-01 12:00:00|BATCH_001|failed   |[invalid_transaction_date]           |1         |
|T004          |-25.0 |2026-05-02 08:30:00|BATCH_002|failed   |[missing_customer_id, invalid_amo

### Example 6: Split cleaned and rejected records

ตอนนี้เราจะรวม DQ issues กับ dedup issue แล้วแยก output เป็น 2 ชุด:

- `df_cleaned_transactions`: record ที่ไม่มี DQ issue และเป็น latest version
- `df_rejected_transactions`: record ที่มี DQ issue หรือเป็น older duplicate version

แนวคิดสำคัญคือ output rejected ควรเก็บเหตุผลไว้ ไม่ใช่แค่เก็บ record ที่ fail แบบไม่มี context

In [8]:
df_final_status = (
    df_ranked
    .withColumn(
        "dedup_issue",
        F.when(F.col("dedup_rank") > 1, F.lit("older_duplicate_version"))
    )
    .withColumn(
        "final_issues",
        F.expr("filter(concat(dq_issues, array(dedup_issue)), x -> x is not null)")
    )
    .withColumn(
        "final_status",
        F.when(F.size(F.col("final_issues")) == 0, F.lit("clean")).otherwise(F.lit("rejected"))
    )
    .withColumn("processed_at", F.current_timestamp())
)

cleaned_columns = [
    "transaction_id",
    "customer_id",
    "customer_name",
    "email",
    "phone",
    "transaction_date",
    "amount",
    "status",
    "payment_method",
    "source_system",
    "batch_id",
    "updated_at",
    "device_platform",
    "device_app_version",
    "city",
    "country",
    "promo_code",
    "processed_at",
]

rejected_columns = [
    "transaction_id",
    "customer_id",
    "customer_name",
    "email",
    "phone",
    "transaction_date",
    "amount",
    "status",
    "payment_method",
    "source_payload_json",
    "payload_parse_status",
    "source_system",
    "batch_id",
    "updated_at",
    "dedup_rank",
    "final_issues",
    "processed_at",
]

df_cleaned_transactions = (
    df_final_status
    .filter(F.col("final_status") == "clean")
    .select(*cleaned_columns)  # * unpacks list into multiple column-name arguments
)

df_rejected_transactions = (
    df_final_status
    .filter(F.col("final_status") == "rejected")
    .select(*rejected_columns)  # * unpacks list into multiple column-name arguments
)

print("Cleaned rows:", df_cleaned_transactions.count())
print("Rejected rows:", df_rejected_transactions.count())

df_cleaned_transactions.orderBy("transaction_id").show(truncate=False)
df_rejected_transactions.orderBy("transaction_id", "dedup_rank").show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 10, Finished, Available, Finished, False)

Cleaned rows: 4
Rejected rows: 8
+--------------+-----------+-------------+-----------------+------------+----------------+------+-------+--------------+-------------+---------+-------------------+---------------+------------------+----------+-------+----------+-------------------------+
|transaction_id|customer_id|customer_name|email            |phone       |transaction_date|amount|status |payment_method|source_system|batch_id |updated_at         |device_platform|device_app_version|city      |country|promo_code|processed_at             |
+--------------+-----------+-------------+-----------------+------------+----------------+------+-------+--------------+-------------+---------+-------------------+---------------+------------------+----------+-------+----------+-------------------------+
|T001          |001        |Alice Wong   |alice@example.com|+66811111111|2026-05-01      |1200.5|success|credit_card   |mobile_app   |BATCH_001|2026-05-01 10:00:00|ios            |1.2.0             |

### Example 7: Write cleaned and rejected outputs to Lakehouse tables

ถ้า Notebook attach กับ Fabric Lakehouse แล้ว สามารถ write output เป็น managed Lakehouse tables ได้ด้วย `saveAsTable`

ตารางที่สร้างในตัวอย่างนี้:

- `day12_cleaned_transactions`
- `day12_rejected_transactions`

ถ้ายังไม่ได้ attach Lakehouse หรือ user ไม่มี permission cell นี้อาจ fail ได้ ให้ attach Lakehouse ก่อนแล้วค่อย run ใหม่

In [9]:
df_cleaned_transactions.write.mode("overwrite").format("delta").saveAsTable("day12_cleaned_transactions")
df_rejected_transactions.write.mode("overwrite").format("delta").saveAsTable("day12_rejected_transactions")

spark.read.table("day12_cleaned_transactions").orderBy("transaction_id").show(truncate=False)
spark.read.table("day12_rejected_transactions").orderBy("transaction_id", "dedup_rank").show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 11, Finished, Available, Finished, False)

+--------------+-----------+-------------+-----------------+------------+----------------+------+-------+--------------+-------------+---------+-------------------+---------------+------------------+----------+-------+----------+--------------------------+
|transaction_id|customer_id|customer_name|email            |phone       |transaction_date|amount|status |payment_method|source_system|batch_id |updated_at         |device_platform|device_app_version|city      |country|promo_code|processed_at              |
+--------------+-----------+-------------+-----------------+------------+----------------+------+-------+--------------+-------------+---------+-------------------+---------------+------------------+----------+-------+----------+--------------------------+
|T001          |001        |Alice Wong   |alice@example.com|+66811111111|2026-05-01      |1200.5|success|credit_card   |mobile_app   |BATCH_001|2026-05-01 10:00:00|ios            |1.2.0             |Bangkok   |TH     |WELCOME10 |

## Quick recap

| Question | Answer |
|---|---|
| Raw-to-cleaned flow เริ่มจากอะไร? | เริ่มจากเข้าใจ raw schema และ normalize column names ก่อน |
| ทำไมต้อง clean string ก่อน cast? | เพราะ raw value อาจมี comma, currency symbol, blank string หรือ case ไม่สม่ำเสมอ |
| Nested JSON column parse ยังไง? | ใช้ `from_json` พร้อม explicit `StructType` schema |
| ทำไมต้องใช้ `_corrupt_record` ตอน parse JSON? | เพื่อเก็บ malformed JSON payload ไม่ให้กลายเป็น null fields เงียบ ๆ |
| Bad records ควรทำยังไง? | ไม่ควร drop เงียบ ๆ ควรสร้าง DQ issues และแยก rejected output |
| Duplicate updates เลือก record ล่าสุดยังไง? | ใช้ window `partitionBy(business_key).orderBy(updated_at desc)` และ `row_number()` |
| Cleaned table ควรตรวจอะไรหลัง write? | row count, schema, sample records และ rejected issue summary |

## Exercises

### Exercise 1: Create rejected issue summary

สร้าง summary ว่าแต่ละ DQ / dedup issue มีจำนวนกี่ records

Requirements:

- ใช้ `df_rejected_transactions`
- explode `final_issues`
- group by issue
- sort จากจำนวนมากไปน้อย

Expected result:

- เห็น issue เช่น `invalid_amount`, `invalid_transaction_date`, `older_duplicate_version`
- ใช้ตรวจได้ว่า rejected records มาจากปัญหาอะไรบ้าง

In [10]:
df_rejected_issue_summary = (
    df_rejected_transactions
    .select(F.explode("final_issues").alias("issue"))
    .groupBy("issue")
    .agg(F.count("*").alias("record_count"))
    .orderBy(F.desc("record_count"), "issue")
)

df_rejected_issue_summary.show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 12, Finished, Available, Finished, False)

+------------------------+------------+
|issue                   |record_count|
+------------------------+------------+
|invalid_amount          |3           |
|invalid_payload_json    |1           |
|invalid_status          |1           |
|invalid_transaction_date|1           |
|missing_customer_id     |1           |
|missing_transaction_id  |1           |
|older_duplicate_version |1           |
+------------------------+------------+



### Exercise 2: Create source system quality summary

สร้าง summary ว่าแต่ละ `source_system` มี clean/rejected records เท่าไร

Requirements:

- ใช้ `df_final_status`
- group by `source_system` และ `final_status`
- count records
- sort ตาม `source_system`, `final_status`

Expected result:

- เห็นภาพว่า source ไหนมี rejected records มากกว่า source อื่น
- เป็น pattern เบื้องต้นสำหรับ monitoring data quality ตาม source

In [14]:
df_source_quality_summary = (
    df_final_status
    .groupBy("source_system", "final_status")
    .agg(F.count("*").alias("record_count"))
    .orderBy("source_system", "final_status")
)

df_source_quality_summary.show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 16, Finished, Available, Finished, False)

+-------------+------------+------------+
|source_system|final_status|record_count|
+-------------+------------+------------+
|call_center  |clean       |1           |
|mobile_app   |clean       |2           |
|mobile_app   |rejected    |5           |
|pos          |rejected    |1           |
|web_portal   |clean       |1           |
|web_portal   |rejected    |2           |
+-------------+------------+------------+



### Exercise 3: Build daily revenue summary from cleaned records

สร้าง summary รายวันจาก cleaned records เฉพาะ transaction ที่ `status = success`

Requirements:

- ใช้ `df_cleaned_transactions`
- filter `status = success`
- group by `transaction_date`
- สรุป `transaction_count` และ `total_amount`
- write เป็น table `day12_daily_revenue_summary`

Expected result:

- ได้ daily summary จากเฉพาะ clean records
- ไม่เอา rejected records ไปปนใน reporting output

In [15]:
df_daily_revenue_summary = (
    df_cleaned_transactions
    .filter(F.col("status") == "success")
    .groupBy("transaction_date")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
    )
    .orderBy("transaction_date")
)

df_daily_revenue_summary.show(truncate=False)

df_daily_revenue_summary.write.mode("overwrite").format("delta").saveAsTable("day12_daily_revenue_summary")
spark.read.table("day12_daily_revenue_summary").show(truncate=False)

StatementMeta(, d38a6cc3-9dfd-4703-9980-4500187fb29c, 17, Finished, Available, Finished, False)

+----------------+-----------------+------------+
|transaction_date|transaction_count|total_amount|
+----------------+-----------------+------------+
|2026-05-01      |1                |1200.5      |
|2026-05-04      |1                |2500.0      |
|2026-05-05      |2                |1500.0      |
+----------------+-----------------+------------+

+----------------+-----------------+------------+
|transaction_date|transaction_count|total_amount|
+----------------+-----------------+------------+
|2026-05-01      |1                |1200.5      |
|2026-05-04      |1                |2500.0      |
|2026-05-05      |2                |1500.0      |
+----------------+-----------------+------------+



## Common mistakes

1. **รีบ cast raw values ก่อน clean string**

   ถ้า `amount` มี comma หรือ currency text เช่น `THB 2,500.00` การ cast เป็น double ทันทีอาจได้ `null` หรือ error ขึ้นกับ runtime/config ควร clean format ก่อน

2. **Drop bad records เงียบ ๆ**

   การ filter records ทิ้งโดยไม่เก็บเหตุผลทำให้ debug ยากมาก ควรมี DQ issue หรือ rejected output อย่างน้อยใน learning lab

3. **Deduplicate ก่อน parse `updated_at` ให้ถูกต้อง**

   ถ้า `updated_at` ยังเป็น string หลาย format การ order เพื่อหา latest record อาจผิด ควร parse เป็น TimestampType ก่อนใช้ window

4. **เช็ก JSON parsing ด้วย `payload.isNull()` อย่างเดียว**

   - ใน lab นี้มี record ที่ `source_payload_json` เป็น `not_json` ซึ่งเป็น malformed JSON
   
   - เมื่อใช้ `from_json` แบบ default โดยไม่ได้กำหนด JSON parsing options เพิ่ม ผลลัพธ์ของ record นี้ไม่ใช่ `payload = null` ทั้งก้อน แต่เป็น `payload = {NULL, NULL, NULL}` ซึ่งเป็น struct ที่ field ข้างในเป็น `null` ทั้งหมด เนื่องจาก Spark สร้างผลลัพธ์ตาม `payload_schema` ได้ แต่ไม่สามารถเติมค่าจาก malformed JSON ได้

   - ถ้าเช็กแค่ `payload.isNull()` จะจับเคสนี้ไม่ได้ เพราะตัว `payload` ยังเป็น struct อยู่ แม้ field ข้างในจะเป็น `null` ทั้งหมดก็ตาม

   - ใน lab นี้จึงใช้ `PERMISSIVE` mode ร่วมกับ `_corrupt_record` เพื่อเก็บ malformed JSON payload และใช้ `payload._corrupt_record` ในการระบุ `invalid_json` แทน

5. **ตีความ `missing_payload` เป็น bad record เสมอ**

   `missing_payload` ไม่จำเป็นต้องเป็น DQ failure เสมอไป ถ้า payload เป็น optional enrichment data เช่น device, location หรือ promo แต่ `invalid_json` ควรถูก flag เพราะ source ส่ง payload มาแล้ว parse ไม่ได้

6. **ใช้ `show()` อย่างเดียวแล้วคิดว่า pipeline ถูกแล้ว**

   ควรตรวจ `.count()`, issue summary, clean/rejected split และ sample records หลายมุม ไม่ใช่ดูแค่ preview 20 rows


## Expected Output / Success Criteria

เมื่อจบ Day 12 ควรทำได้:

- อธิบาย raw-to-cleaned transformation flow ได้ตั้งแต่ raw input จนถึง cleaned/rejected output
- normalize column names จาก raw source ให้เป็น `snake_case` ได้
- standardize string fields เช่น `status`, `email`, `payment_method`, `phone` ได้
- parse `amount`, `transaction_date` และ `updated_at` จาก raw string เป็น data type ที่เหมาะสมได้
- parse nested JSON string ด้วย `from_json` และ flatten nested fields ออกมาเป็น columns ได้
- ใช้ `PERMISSIVE` mode ร่วมกับ `_corrupt_record` เพื่อจับ malformed JSON payload เช่น `not_json` ได้
- แยก `payload_parse_status` เป็น `missing_payload`, `invalid_json` และ `parsed` ได้
- อธิบายได้ว่า `missing_payload` อาจไม่ใช่ DQ failure เสมอไป ถ้า payload เป็น optional enrichment data
- สร้าง `dq_issues` และ `dq_status` เพื่อบอกว่าแต่ละ record มี issue อะไรได้
- ใช้ window function เพื่อ keep latest transaction ต่อ `transaction_id` ได้
- แยก clean records กับ rejected records ได้โดยไม่ drop bad records เงียบ ๆ
- write cleaned และ rejected output เป็น Lakehouse tables ได้ ถ้า attach Lakehouse เรียบร้อย
- สร้าง rejected issue summary และ source quality summary เพื่อตรวจสอบผลลัพธ์ได้


## Final takeaway

Raw-to-cleaned mini challenge วันนี้คือจุดที่เริ่มรวม PySpark skill หลายเรื่องเข้าด้วยกัน

> Cleaned pipeline ที่ดีต้องไม่ใช่แค่ทำให้ output ดูสวย แต่ต้องอธิบายได้ว่า raw data ถูกเปลี่ยนอย่างไร record ไหนผ่าน record ไหนไม่ผ่าน และเพราะอะไร

Mindset ที่ควรจำ:

- raw values ต้องถูก inspect ก่อน transform
- casting ควรตามหลัง standardization
- malformed JSON payload ควรถูก flag และเก็บไว้ตรวจสอบได้
- DQ issue ควรมองเห็นได้ใน output
- duplicate logic ต้องอิง business key และ timestamp ที่ parse แล้ว
- cleaned output กับ rejected output ควรตรวจ row count และ reason summary ทุกครั้ง

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day12_cleaned_transactions")
# spark.sql("DROP TABLE IF EXISTS day12_rejected_transactions")
# spark.sql("DROP TABLE IF EXISTS day12_daily_revenue_summary")